[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ciri/iese-dsfb/blob/main/notebooks/420-Airline-Text-Analytics.ipynb)

# Airline Text Analytics

As part of an effort to better understand passenger satisfaction within the Star Alliance network, we analyze customer reviews from thousands of airline passengers. The goal is to identify how factors like sentiment, seat type, travel purpose, and specific routes relate to overall recommendation rates.

The dataset includes 23,000+ reviews, with key variables such as:

- `Airline Name`: the name of the airline.
- `Route`: the flight route taken by the passenger.
- `Review`: free-text review written by the traveler.
- `Type Of Traveller`: e.g., Solo Leisure, Business, Couple.
- `Recommended`: whether the traveler recommends the airline (yes/no).
- `Seat Type`: seat class (e.g., Economy, Business).

Data Source: [Airlinequality.com](https://www.airlinequality.com/) via [Kaggle](https://www.kaggle.com/datasets/khushipitroda/airline-reviews/data)

## Part I: Working with Text Data

### Strings

Text data always starts out in the form of a **string** — a sequence of characters. This includes alphanumeric characters, spaces, punctuation, and symbols. Each character is represented internally as a number (the ASCII/UTF-8 encoding), but we rarely need to think about that in practice.

### Python string methods

Python has a rich set of methods for manipulating strings:

| Method | What it does |
|---|---|
| `lower()` / `upper()` | Change case |
| `replace(old, new)` | Find and replace |
| `split(sep)` | Split into a list of parts |
| `join(list)` | Join a list back into a string |
| `count(pattern)` | Count occurrences |

In [2]:
name = 'Enric Junque de Fortuny'

print(name.upper())
print(name.lower())
print(name.split(' '))
print('---'.join(name.split(' ')))

ENRIC JUNQUE DE FORTUNY
enric junque de fortuny
['Enric', 'Junque', 'de', 'Fortuny']
Enric---Junque---de---Fortuny


### Strings as lists

Both lists and strings are **sequences**, so they share some core behaviours. A string can be thought of as a list of characters. You can use `len()` and slicing just like a list:

In [3]:
school = 'IESE Business School'
print(len(school))
print(school[:4])    # first 4 characters
print(school[-6:])   # last 6 characters

20
IESE
School


## Airline Reviews Dataset

Let's load the data. I've already partially cleaned it up for you (no duplicates). You can download it from [here](https://raw.githubusercontent.com/ciri/iese-dsfb/refs/heads/main/resources/text/Airline_Reviews.csv).

In [4]:
import pandas as pd

pd.set_option('display.max_colwidth', None)

## Use your local copy:
df = pd.read_csv('../resources/text/Airline_Reviews.csv')[['Airline Name','Route','Review','Type Of Traveller','Recommended','Seat Type']]
## Or download every time:
# df = pd.read_csv('https://raw.githubusercontent.com/ciri/iese-dsfb/refs/heads/main/resources/text/Airline_Reviews.csv')[['Airline Name','Route','Review','Type Of Traveller','Recommended','Seat Type']]

df.head()

,Airline Name,Route,Review,Type Of Traveller,Recommended,Seat Type
0,AB Aviation,Moroni to Moheli,"Moroni to Moheli. Turned out to be a pretty decent airline. Online booking worked well, checkin and boarding was fine and the plane looked well maintained. Its a very short flight - just 20 minutes or so so i didn't expect much but they still managed to hand our a bottle of water and some biscuits which i though was very nice. Both flights on time.",Solo Leisure,yes,Economy Class
1,AB Aviation,Moroni to Anjouan,"Moroni to Anjouan. It is a very small airline. My ticket advised me to turn up at 0800hrs which I did. There was confusion at this small airport. I was then directed to the office of AB Aviation which was still closed. It opened at 0900hrs and I was told that the flight had been put back to 1300hrs and that they had tried to contact me. This could not be true as they did not have my phone number. I was with a local guide and he had not been informed either. I presume that I was bumped off. The later flight did operate but as usual, there was confusion at check-in. The flight was only 30mins and there were no further problems. Not a good airline but it is the only one for Comoros.",Solo Leisure,no,Economy Class
2,AB Aviation,Anjouan to Dzaoudzi,"Anjouan to Dzaoudzi. A very small airline and the only airline based in Comoros. Check-in was disorganised because of locals with big packages and disinterested staff. The flight was fortunately short (30 mins). Took off on time and landed on time. With a short flight like there was of course no in-flight entertainment nor cabin service except for biscuits and a bottle of water, which was quite nice!",Solo Leisure,no,Economy Class
3,Adria Airways,Frankfurt to Pristina,"Please do a favor yourself and do not fly with Adria. On the route from Munich to Pristina in July 2019 they lost my luggage and for 10 days in a row, despite numerous phone calls they were not able to locate it. 11 days later the luggage arrived at the destination completely ruined. Applying for compensation, they ignored my request. Foolishly again, I booked another flight with them (345 euros) Frankfurt - Pristina in September 2019. They cancelled the flight with no reason 24 hours before the departure. Desperate phone calls to customer service to get anything (rerouting, compensation, etc) were not responded. I will never fly again with Adria. What a disgrace! Shame on you Adria for constantly deceiving your customers.",Solo Leisure,no,Economy Class
4,Adria Airways,Sofia to Amsterdam via Ljubljana,"Do not book a flight with this airline! My friend and I should have returned from Sofia to Amsterdam on September 22 and 3 days before, they sent us an SMS informing the flight was cancelled. For 3 straight days we tried to reach the airline and the web agent (e-dreams) and we did not get a solution. Finally, 18 hours before our cancelled flight time, and after 35 minutes on a call (waiting), the airline was able to get us on a flight with Lufthansa. Do not book Adria Airways, it is unreliable and in our case, it ruined our last days of holidays since we needed to be on the phones all day.",Couple Leisure,no,Economy Class


### Exploring the dataset

There is not much to `describe()` here since most columns are text. Let's use `.value_counts()` to count categories, and `.groupby()` to aggregate.

In [5]:
df['Seat Type'].value_counts()

Seat Type
Economy Class      19129
Business Class      2097
Premium Economy      646
First Class          186
Name: count, dtype: int64

In [7]:
df.groupby('Seat Type').count()

,Airline Name,Route,Review,Type Of Traveller,Recommended
Seat Type,,,,,
Business Class,2097,1806,2097,1808,2097
Economy Class,19129,16747,19129,16829,19129
First Class,186,168,186,168,186
Premium Economy,646,608,646,613,646


**You try it**

1. Create a dummy for whether or not a passenger recommends a particular flight/airline.
2. Then, show the *recommendation rate* per type of traveler.

### Exploring the review text

When a pandas column contains strings, we access its **string methods** via `.str`. These are vectorized — they apply to every row at once. The syntax is `series.str.method(args)`.

For example, `.str.contains("pattern")` returns a Boolean series — True wherever the pattern appears:

In [ ]:
f_has_space = df.Review.str.lower().str.contains("space")
df[f_has_space].head(3)

In [ ]:
# Let's read one full review
print(df.loc[41].Review)

**You try it**

1. Use `str.contains` to find the number of reviews which mention the following keywords: `'Crew'`, `'Bag'`, `'Passenger'`.
2. Calculate the total count across all 3 keywords.
3. Does review 186 from Aegean Airlines contain the word crew?

### Topic Detection: Frequent Word Analysis

Advanced techniques like topic modeling (LDA) or transformer-based classification exist, but a simple and quick method is **word frequency analysis**. It gives a fast sense of recurring topics — service, food, delays, seats — without any machine learning.

In [ ]:
sample = df.sample(1000, random_state=42)
all_words = sample.Review.str.lower().str.split().sum()

pd.Series(all_words).value_counts().head(20)[::-1].plot.barh(color='orange');

The top terms are mostly **stopwords** — words that carry no meaning (the, a, I, was ...). We can filter them out using a standard list from NLTK:

In [ ]:
#! pip install nltk
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

stop = set(stopwords.words('english'))

filtered_words = [w for w in all_words if w not in stop]
pd.Series(filtered_words).value_counts().head(20)[::-1].plot.barh(color='orange');

**You try it**

The bar chart above gives you the top words — but a wordcloud can make this more striking. Ask your favourite LLM to generate the code for you. Something like:

> *"Generate Python code to create a wordcloud from a Python list of strings called `filtered_words`"*

Paste the generated code in the cell below and run it.

In [ ]:
# paste LLM-generated wordcloud code here

---
## Part II: Sentiment Analysis

Word frequency tells us **what** passengers write about. But it treats every word in isolation — it can't distinguish *"good food"* from *"not good food"*, or *"no delays"* from *"constant delays"*. Is looking at individual words really enough?

For that we need **sentiment analysis** — an NLP technique that quantifies the emotion or opinion expressed in text. It's a staple of marketing analytics: understanding how customers feel about a brand, product, or service from reviews, social media, and surveys.

We'll try three approaches, each more powerful — and more complex — than the last:

| Approach | Method | Speed | Accuracy |
|---|---|---|---|
| 1. Ad-hoc | Keyword matching | Instant | Low |
| 2. Lexicon-based | VADER dictionary | Fast | Medium |
| 3. Machine Learning | Transformer model | Slower | High |

### Approach 1: Ad-hoc (Keyword Matching)

The simplest method: check if positive or negative words appear in the review.

In [ ]:
good = df.Review.str.lower().str.contains('good')
bad  = df.Review.str.lower().str.contains('bad')

print(f"Reviews containing 'good': {good.mean():.1%}")
print(f"Reviews containing 'bad':  {bad.mean():.1%}")

**You try it**

Implement the sentiment scoring function. Define the sentiment of review $i$ as:
$$S_i = G_i - B_i$$
where $G_i = 1$ if "good" appears, $B_i = 1$ if "bad" appears.

Plot the distribution of sentiment scores as a histogram.

Here is an alternative way to write this using `.apply()`, which applies a function row-by-row. This pattern becomes essential for more complex scoring later:

In [ ]:
def sentiment(review):
    is_good = 'good' in review.lower()
    is_bad  = 'bad'  in review.lower()
    return is_good - is_bad

df['simple_sentiment'] = df.Review.apply(sentiment)
df.simple_sentiment.hist(bins=5);

**Litmus test**

Does our simple sentiment score actually correlate with whether passengers recommend the airline?

In [ ]:
df["Recommended01"] = (df.Recommended == 'yes').astype(int)
df[['simple_sentiment','Recommended01']].corr()

### Approach 2: Lexicon-Based (VADER)

The **VADER** (Valence Aware Dictionary and sEntiment Reasoner) lexicon was developed by Hutto & Gilbert (2014) at Georgia Tech. They had human annotators rate thousands of words for sentiment intensity, then built a dictionary capturing both polarity (positive/negative) and strength. It handles informal text, punctuation emphasis (!), and capitalization well.

In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

In [ ]:
# Peek at the lexicon — each word has a valence score
print(sia.lexicon.get("amazing"))
print(sia.lexicon.get("terrible"))
print(sia.lexicon.get("okay"))

In [ ]:
# polarity_scores returns neg/neu/pos/compound — compound is the overall score (-1 to +1)
sia.polarity_scores('I really loved the snacks on the flight!')

In [ ]:
from tqdm import tqdm
tqdm.pandas()

def get_sentiment(review):
    return sia.polarity_scores(review)['compound']

df['sentiment'] = df['Review'].progress_apply(get_sentiment)

**You try it**

1. Show the 2 most positive and 2 most negative reviews according to VADER.
2. Run the correlation test: does VADER sentiment correlate better with `Recommended01` than the ad-hoc approach?

### Approach 3: Machine Learning (Transformer)

VADER is rule-based and can't handle negation ("not bad" still scores negatively). Transformer models like **DistilBERT** are trained on millions of labelled sentences and understand context much better. The model below is already fine-tuned for sentiment — no training needed.

> Note: this may be slow on older computers. If it doesn't run, use Google Colab.

In [ ]:
#! pip install transformers torch
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=512
)

def scored_sentiment(text):
    result = sentiment_model(text)[0]
    return result['score'] if result['label'] == 'POSITIVE' else -result['score']

# Quick demo
print(scored_sentiment('Best. Flight. Ever.'))
print(scored_sentiment('Worst experience of my life. Never again.'))

**Going further: emotion detection**

Binary positive/negative is useful but coarse. The model below — trained by your professor on the [GoEmotions dataset](https://huggingface.co/datasets/google-research-datasets/go_emotions) — classifies text into 28 fine-grained emotions (joy, anger, disappointment, admiration ...). It's based on ModernBERT, a 2024 architecture, and available on [HuggingFace](https://huggingface.co/cirimus/modernbert-base-go-emotions).

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="cirimus/modernbert-base-go-emotions",
    return_all_scores=True
)

text = "The crew was amazing but my seat was incredibly uncomfortable for a 12-hour flight."
predictions = classifier(text)

top_5 = sorted(predictions[0], key=lambda x: x['score'], reverse=True)[:5]
print("Top 5 emotions detected:")
for pred in top_5:
    print(f"  {pred['label']:15s}: {pred['score']:.3f}")

### Who Are the Stars in Star Alliance?

Now that we have sentiment scores, let's rank airlines. We'll filter to Star Alliance members for a focused comparison.

In [ ]:
df.groupby('Airline Name')['sentiment'].mean().sort_values().plot.barh(
    color='orange', figsize=(8,12)
);

In [ ]:
star_alliance_members = [
    'Aegean Airlines','Air Canada','Air China','Air India',
    'ANA All Nippon Airways','Asiana Airlines','Austrian Airlines',
    'Avianca','Brussels Airlines','Copa Airlines','Croatia Airlines',
    'Egyptair','Ethiopian Airlines','Eva Air','LOT Polish Airlines',
    'Lufthansa','Scandinavian Airlines','Shenzhen Airlines',
    'Singapore Airlines','South African Airways',
    'Swiss International Air Lines','TAP Portugal',
    'Thai Airways','Turkish Airlines','United Airlines'
]

f_star = df['Airline Name'].isin(star_alliance_members)
df[f_star].groupby('Airline Name')['sentiment'].mean().sort_values().plot.barh(
    color='steelblue', title='Star Alliance — Average VADER Sentiment'
);

> What follow-up analysis would you do if you worked at Star Alliance?

---
## Part III: Aspect-Based Analysis with LLMs

Overall sentiment tells you whether a review is positive or negative. But an airline category manager needs sharper answers: *How is the crew rated? What do passengers say about luggage? How does food score on long-haul routes?*

This is **aspect-based sentiment analysis** — and it's where LLMs shine. Rather than training a separate model for each aspect (crew, food, luggage, punctuality ...), you just change the prompt.

You have two options for any custom NLP task:
1. Fine-tune your own model (requires labelled data — take the ML class)
2. **Use an LLM to annotate your data** — powerful, fast, and requires no training data

Let's try the second with Google Gemini.

**Setup:**
1. Create a [Google account](https://accounts.google.com/) or log in.
2. Create a free [Gemini API key](https://aistudio.google.com/app/apikey).
3. Install the SDK: `pip install -q -U google-genai`

In [ ]:
API_KEY = '...'  # paste your key here

In [ ]:
# ! pip install -q -U google-genai
from google import genai

client = genai.Client(api_key=API_KEY)

response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Explain in one number what the meaning of life is."
)
print(response.text)

### Scoring an aspect: crew

We'll start by scoring the **crew** aspect — one of the most frequently mentioned in airline reviews. The key idea: write a prompt that extracts a structured score from free text. First test your prompt manually in your favourite LLM chat interface, then automate it below.

In [ ]:
df_ana = df[df['Airline Name'] == 'ANA All Nippon Airways']
df_ana[['Airline Name','Review']].head()

In [ ]:
PROMPT = """
You're a world-class customer service analyst who is analyzing airline reviews.
Score the crew on the flight (friendliness, professionalism, helpfulness) from 0 to 10.
If crew is not mentioned, return 'N/A'.

Also extract the most relevant passage that supports your score.

Format your response as:

<score>YOUR_SCORE_HERE</score><passage>YOUR_PASSAGE_HERE</passage>

Important: only reply with the required tags and scores. Don't add anything else.
"""

def score_crew(review_text):
    prompt = PROMPT + "\nReview:" + review_text
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt
    )
    return response.text.strip()

score_crew('the crew was amazing. the food was good too.')

**You try it**

1. Apply `score_crew()` to `df_sample` below (50 rows — don't run on the full dataset yet, it uses API quota).
2. Parse the output: create a `CrewScore` column and a `Rationale` column.
3. **Bonus:** change the `PROMPT` to score a *different* aspect — luggage handling, food quality, or punctuality. How little do you need to change to pivot to a completely different analysis?

Hints:
- Use `.apply(score_crew)` to get a column of raw LLM output strings.
- Use string splitting on `<score>` and `<passage>` tags to extract the values.
- LLMs sometimes don't follow instructions — filter out rows where the tags are missing.

In [ ]:
df_sample = df.sample(50, random_state=42)
# llm_scores = df_sample['Review'].apply(score_crew)
# ...